# CastStar on GIFT-Eval

This notebook reproduces the CastStar submission from frozen inference assets. It downloads the artifact bundle, runs the CastStar predictor, evaluates all 97 GIFT-Eval configurations, and writes a submission-format CSV.

## Setup

1. Install GIFT-Eval from this repository.
2. Download the Salesforce/GiftEval dataset.
3. Set `GIFT_EVAL_DATA` below to the downloaded dataset directory.
4. Install the additional dependencies in the next cell.


In [ ]:
%pip install -q xgboost huggingface_hub


## Download and configure the inference assets


In [ ]:
from pathlib import Path
from types import SimpleNamespace

from huggingface_hub import snapshot_download

HF_REPO_ID = "USTC-AGI/CastStar"
GIFT_EVAL_DATA = Path("/path/to/GiftEval")
GIFT_EVAL_REPO = Path("..").resolve()
ARTIFACTS_DIR = Path("./caststar_artifacts")
DOWNLOAD_FROM_HUB = True

if DOWNLOAD_FROM_HUB:
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="model",
        local_dir=ARTIFACTS_DIR,
    )

NOTEBOOK_ARGS = SimpleNamespace(
    manifest=ARTIFACTS_DIR / "manifest.json",
    gift_eval_data=GIFT_EVAL_DATA,
    gift_eval_repo=GIFT_EVAL_REPO,
    artifact_root=ARTIFACTS_DIR / "test",
    test_feature_root=ARTIFACTS_DIR / "test" / "input_features",
    test_prediction_root=ARTIFACTS_DIR / "test" / "predictions",
    all_model_feature_root=ARTIFACTS_DIR / "test" / "predicted_features",
    pre_model_root=ARTIFACTS_DIR / "models" / "pre",
    post_model=ARTIFACTS_DIR / "models" / "post" / "pair_difference.ubj",
    class_bias=ARTIFACTS_DIR / "calibration" / "bucket_class_biases.csv",
    reference_metrics=None,
    output=Path("../results/CastStar/reproduced_all_results.csv"),
    model_name="CastStar",
    post_output_mode="top1_weight",
    reference_method="",
    expected_composite=0.5642995568949057,
    device="cpu",
    limit_configs=None,
    skip_reference_check=True,
)


## CastStar predictor and GIFT-Eval evaluation


In [ ]:
#!/usr/bin/env python3
"""Reproduce a CastStar GIFT-Eval result from frozen inference assets.

The file is self-contained: it does not import any function from ARFT_TS. It
loads a six-class Pre XGBoost gate, bucket biases, a binary or continuous Post
XGBoost gate, cached base forecasts/features, and the official GIFT-Eval data.
"""

from __future__ import annotations

import argparse
import json
import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Iterator

import numpy as np
import pandas as pd
import xgboost as xgb
from datasets import load_from_disk


MODEL_NAME = "CastStar"
EXPECTED_COMPOSITE = 0.5642995568949057
REFERENCE_METRIC_ATOL = 2e-6
QUANTILE_LEVELS = np.asarray(
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    dtype=np.float64,
)

METRIC_COLUMNS = [
    "MSE[mean]",
    "MSE[0.5]",
    "MAE[0.5]",
    "MASE[0.5]",
    "MAPE[0.5]",
    "sMAPE[0.5]",
    "MSIS",
    "RMSE[mean]",
    "NRMSE[mean]",
    "ND[0.5]",
    "mean_weighted_sum_quantile_loss",
]
OUTPUT_COLUMNS = [
    "dataset",
    "model",
    *[f"eval_metrics/{column}" for column in METRIC_COLUMNS],
    "domain",
    "num_variates",
]


def official_dataset_name(dataset_config: str) -> str:
    """Use the frequency labels used by GIFT-Eval result CSV files."""
    dataset, frequency, term = dataset_config.split("/")
    if frequency.startswith("W-"):
        frequency = "W"
    elif frequency.startswith("Q-"):
        frequency = "Q"
    elif frequency.startswith("A-"):
        frequency = "A"
    return f"{dataset}/{frequency}/{term}"


FORECAST_FEATURE_NAMES = [
    "alpha",
    "arch_lm",
    "beta",
    "crossing_points",
    "curvature",
    "diff1_acf1",
    "diff1_acf10",
    "diff1x_pacf5",
    "diff2_acf1",
    "diff2_acf10",
    "diff2x_pacf5",
    "e_acf1",
    "e_acf10",
    "entropy",
    "flat_spots",
    "hurst",
    "hw_alpha",
    "hw_beta",
    "hw_gamma",
    "linearity",
    "lumpiness",
    "nonlinearity",
    "nperiods",
    "peak",
    "seas_acf1",
    "seas_pacf",
    "seasonal_period",
    "seasonal_strength",
    "series_length",
    "spike",
    "stability",
    "trend",
    "trough",
    "unitroot_kpss",
    "unitroot_pp",
    "x_acf1",
    "x_acf10",
    "x_pacf5",
]
PAIR_FEATURE_NAMES = [
    "median_mae_scaled",
    "median_rmse_scaled",
    "median_max_abs_scaled",
    "median_mean_signed_scaled",
    "median_correlation",
    "top1_first_jump_scaled",
    "top2_first_jump_scaled",
    "top1_mean_offset_scaled",
    "top2_mean_offset_scaled",
    "top1_interval_width_scaled",
    "top2_interval_width_scaled",
    "interval_width_difference_scaled",
    "top1_median_volatility_scaled",
    "top2_median_volatility_scaled",
]
RANK_FEATURE_NAMES = [
    "top1_logit",
    "top2_logit",
    "logit_margin",
    "top1_probability",
    "top2_probability",
    "probability_margin",
    "ranking_entropy",
]


@dataclass(frozen=True)
class ConfigSpec:
    dirname: str
    dataset_name: str
    dataset_config: str
    bucket: str
    freq: str
    term: str
    prediction_length: int
    seasonality: int
    num_variates: int
    domain: str
    windows: int
    rows: int
    normalizer_mase: float
    normalizer_crps: float




def parse_args():
    return NOTEBOOK_ARGS


class ArrayQuantilePredictor:
    def __init__(
        self, prediction_length: int, forecasts: np.ndarray
    ) -> None:
        from gluonts.model.forecast import QuantileForecast

        self.prediction_length = int(prediction_length)
        self.forecasts = forecasts
        self.forecast_class = QuantileForecast
        self.forecast_keys = [
            "mean",
            *[str(value) for value in QUANTILE_LEVELS],
        ]

    def predict(
        self, dataset: Iterator[dict], **kwargs
    ) -> Iterator[object]:
        produced = 0
        for produced, entry in enumerate(dataset, start=1):
            if produced > len(self.forecasts):
                raise ValueError("Dataset contains more rows than forecasts")
            quantiles = self.forecasts[produced - 1]
            yield self.forecast_class(
                forecast_arrays=np.concatenate(
                    [quantiles[4:5], quantiles], axis=0
                ),
                forecast_keys=self.forecast_keys,
                start_date=entry["start"] + len(entry["target"]),
                item_id=entry.get("item_id"),
            )
        if produced != len(self.forecasts):
            raise ValueError(
                f"Dataset/forecast rows differ: "
                f"{produced} != {len(self.forecasts)}"
            )


def spec_from_row(row: dict) -> ConfigSpec:
    return ConfigSpec(
        dirname=str(row["dirname"]),
        dataset_name=str(row["dataset_name"]),
        dataset_config=str(row["dataset_config"]),
        bucket=str(row["bucket"]),
        freq=str(row["freq"]),
        term=str(row["term"]),
        prediction_length=int(row["prediction_length"]),
        seasonality=int(row["seasonality"]),
        num_variates=int(row["num_variates"]),
        domain=str(row["domain"]),
        windows=int(row["windows"]),
        rows=int(row["test_rows"]),
        normalizer_mase=float(row["normalizer_mase"]),
        normalizer_crps=float(row["normalizer_crps"]),
    )


def scalar(value: np.ndarray) -> object:
    array = np.asarray(value)
    return array.item() if array.shape == () else array


def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)


def load_biases(
    path: Path, models: list[str]
) -> dict[str, np.ndarray]:
    frame = pd.read_csv(path)
    result = {}
    for bucket, group in frame.groupby("bucket", sort=False):
        values = group.set_index("model")["bias"]
        if set(values.index) != set(models):
            raise ValueError(f"{path}/{bucket}: model vocabulary mismatch")
        result[str(bucket)] = np.asarray(
            [values[model] for model in models], dtype=np.float32
        )
    return result


def load_lookback_frame(
    feature_root: Path,
    spec: ConfigSpec,
    feature_columns: list[str],
    categories: dict[str, list[str]],
) -> pd.DataFrame:
    root = feature_root / spec.dirname
    with np.load(
        root / "test_features.npz", allow_pickle=False
    ) as data:
        frame = pd.DataFrame(
            data["X"],
            columns=[str(value) for value in data["feature_names"]],
        )
    with np.load(
        root / "test_metadata.npz", allow_pickle=False
    ) as data:
        metadata = {
            name: scalar(data[name]) for name in data.files
        }
    frame["seasonality"] = np.float32(metadata["seasonality"])
    frame["prediction_length"] = np.float32(
        metadata["prediction_length"]
    )
    frame["num_variates"] = np.float32(metadata["num_variates"])
    frame["freq"] = str(metadata["freq"])
    frame["domain"] = str(metadata["domain"])
    frame = frame.reindex(columns=feature_columns)
    for column in ("freq", "domain"):
        frame[column] = frame[column].astype(
            pd.CategoricalDtype(categories=categories[column])
        )
    if len(frame) != spec.rows:
        raise ValueError(
            f"{spec.dirname}: feature rows {len(frame)} != {spec.rows}"
        )
    return frame


def predict_pre_ranking(
    booster: xgb.Booster,
    frame: pd.DataFrame,
    model_count: int,
    bias: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    best_iteration = getattr(booster, "best_iteration", None)
    iteration_range = (
        (0, int(best_iteration) + 1)
        if best_iteration is not None
        else (0, 0)
    )
    logits = np.asarray(
        booster.predict(
            xgb.DMatrix(frame, enable_categorical=True),
            output_margin=True,
            iteration_range=iteration_range,
        ),
        dtype=np.float32,
    ).reshape(len(frame), model_count)
    logits += bias[None, :]
    rows = np.arange(len(frame))
    top1 = np.argmax(logits, axis=1)
    remaining = logits.copy()
    remaining[rows, top1] = -np.inf
    top2 = np.argmax(remaining, axis=1)
    probabilities = softmax(logits)
    top1_logit = logits[rows, top1]
    top2_logit = logits[rows, top2]
    top1_probability = probabilities[rows, top1]
    top2_probability = probabilities[rows, top2]
    entropy = -np.sum(
        probabilities
        * np.log(np.maximum(probabilities, 1e-12)),
        axis=1,
    )
    rank_features = np.column_stack(
        [
            top1_logit,
            top2_logit,
            top1_logit - top2_logit,
            top1_probability,
            top2_probability,
            top1_probability - top2_probability,
            entropy,
        ]
    ).astype(np.float32)
    return (
        top1.astype(np.int16),
        top2.astype(np.int16),
        rank_features,
    )


def load_selected_predictions(
    prediction_root: Path,
    models: list[str],
    spec: ConfigSpec,
    selected: np.ndarray,
) -> np.ndarray:
    result = np.full(
        (
            spec.rows,
            len(QUANTILE_LEVELS),
            spec.prediction_length,
        ),
        np.nan,
        dtype=np.float32,
    )
    for model_index in np.unique(selected):
        positions = np.flatnonzero(selected == model_index)
        path = (
            prediction_root
            / models[int(model_index)]
            / spec.dirname
            / "test_predictions.npz"
        )
        with np.load(path, allow_pickle=False) as data:
            predictions = data["predictions"].astype(
                np.float32, copy=False
            )
        expected = (
            spec.rows,
            len(QUANTILE_LEVELS),
            spec.prediction_length,
        )
        if predictions.shape != expected:
            raise ValueError(
                f"{path}: {predictions.shape} != {expected}"
            )
        result[positions] = predictions[positions]
    if not np.isfinite(result).all():
        raise ValueError(f"{spec.dirname}: non-finite base forecast")
    return result


def load_selected_window_features(
    root: Path,
    models: list[str],
    spec: ConfigSpec,
    selected: np.ndarray,
) -> pd.DataFrame:
    output = np.full(
        (spec.rows, len(FORECAST_FEATURE_NAMES)),
        np.nan,
        dtype=np.float32,
    )
    for model_index in np.unique(selected):
        positions = np.flatnonzero(selected == model_index)
        path = (
            root
            / models[int(model_index)]
            / spec.dirname
            / "predicted_window_features.npz"
        )
        with np.load(path, allow_pickle=False) as data:
            indices = data["indices"].astype(np.int32, copy=False)
            names = [str(value) for value in data["feature_names"]]
            values = pd.DataFrame(
                data["X"], columns=names
            ).reindex(columns=FORECAST_FEATURE_NAMES)
            finite = data["prediction_finite"].astype(bool, copy=False)
            errors = data["feature_error"].astype(bool, copy=False)
        if not np.array_equal(indices, np.arange(spec.rows)):
            raise ValueError(f"{path}: row order mismatch")
        if not finite.all() or errors.any():
            raise ValueError(f"{path}: invalid feature asset")
        output[positions] = values.to_numpy(dtype=np.float32)[
            positions
        ]
    return pd.DataFrame(output, columns=FORECAST_FEATURE_NAMES)


def seasonal_scale(context: np.ndarray, seasonality: int) -> float:
    period = max(1, int(seasonality))
    if len(context) <= period:
        return math.nan
    left = context[period:]
    right = context[:-period]
    mask = np.isfinite(left) & np.isfinite(right)
    if not mask.any():
        return math.nan
    value = float(np.mean(np.abs(left[mask] - right[mask])))
    return value if value > 0.0 and np.isfinite(value) else math.nan


def load_context_end_and_scale(
    gift_eval_data: Path, spec: ConfigSpec
) -> np.ndarray:
    values = []
    dataset = load_from_disk(
        str(gift_eval_data / spec.dataset_name)
    ).with_format("numpy")
    for entry in dataset:
        target = np.asarray(entry["target"], dtype=np.float64)
        if target.ndim == 1:
            target = target[None, :]
        for variate in range(spec.num_variates):
            series = target[variate]
            base_offset = (
                len(series)
                - spec.windows * spec.prediction_length
            )
            for window in range(spec.windows):
                start = (
                    base_offset
                    + window * spec.prediction_length
                )
                context = series[:start]
                finite = context[np.isfinite(context)]
                last_value = (
                    float(finite[-1]) if len(finite) else math.nan
                )
                values.append(
                    (
                        last_value,
                        seasonal_scale(context, spec.seasonality),
                    )
                )
    result = np.asarray(values, dtype=np.float64)
    if result.shape != (spec.rows, 2):
        raise ValueError(
            f"{spec.dirname}: contexts {result.shape} "
            f"!= {(spec.rows, 2)}"
        )
    return result


def safe_scaled(
    value: np.ndarray, scale: np.ndarray
) -> np.ndarray:
    return np.divide(
        value,
        scale,
        out=np.full(
            np.broadcast_shapes(value.shape, scale.shape), np.nan
        ),
        where=np.isfinite(scale) & (scale > 0),
    )


def compute_pair_features(
    top1_predictions: np.ndarray,
    top2_predictions: np.ndarray,
    last_value: np.ndarray,
    scale: np.ndarray,
) -> np.ndarray:
    top1_median = top1_predictions[:, 4, :].astype(np.float64)
    top2_median = top2_predictions[:, 4, :].astype(np.float64)
    difference = top1_median - top2_median
    mae = safe_scaled(
        np.nanmean(np.abs(difference), axis=1), scale
    )
    rmse = safe_scaled(
        np.sqrt(np.nanmean(difference**2, axis=1)), scale
    )
    max_abs = safe_scaled(
        np.nanmax(np.abs(difference), axis=1), scale
    )
    mean_signed = safe_scaled(
        np.nanmean(difference, axis=1), scale
    )
    correlation = np.full(
        len(difference), np.nan, dtype=np.float64
    )
    for row_index in range(len(difference)):
        left = top1_median[row_index]
        right = top2_median[row_index]
        mask = np.isfinite(left) & np.isfinite(right)
        if (
            mask.sum() > 1
            and np.std(left[mask]) > 0
            and np.std(right[mask]) > 0
        ):
            correlation[row_index] = np.corrcoef(
                left[mask], right[mask]
            )[0, 1]
    top1_jump = safe_scaled(
        top1_median[:, 0] - last_value, scale
    )
    top2_jump = safe_scaled(
        top2_median[:, 0] - last_value, scale
    )
    top1_offset = safe_scaled(
        np.nanmean(top1_median, axis=1) - last_value, scale
    )
    top2_offset = safe_scaled(
        np.nanmean(top2_median, axis=1) - last_value, scale
    )
    top1_width = safe_scaled(
        np.nanmean(
            top1_predictions[:, 8, :]
            - top1_predictions[:, 0, :],
            axis=1,
        ),
        scale,
    )
    top2_width = safe_scaled(
        np.nanmean(
            top2_predictions[:, 8, :]
            - top2_predictions[:, 0, :],
            axis=1,
        ),
        scale,
    )
    top1_volatility = safe_scaled(
        np.nanmean(
            np.abs(np.diff(top1_median, axis=1)), axis=1
        ),
        scale,
    )
    top2_volatility = safe_scaled(
        np.nanmean(
            np.abs(np.diff(top2_median, axis=1)), axis=1
        ),
        scale,
    )
    return np.column_stack(
        [
            mae,
            rmse,
            max_abs,
            mean_signed,
            correlation,
            top1_jump,
            top2_jump,
            top1_offset,
            top2_offset,
            top1_width,
            top2_width,
            top1_width - top2_width,
            top1_volatility,
            top2_volatility,
        ]
    ).astype(np.float32)


def build_post_frame(
    lookback: pd.DataFrame,
    top1_features: pd.DataFrame,
    top2_features: pd.DataFrame,
    top1_predictions: np.ndarray,
    top2_predictions: np.ndarray,
    top1: np.ndarray,
    top2: np.ndarray,
    rank_features: np.ndarray,
    contexts: np.ndarray,
    models: list[str],
    expected_columns: list[str],
) -> pd.DataFrame:
    first = top1_features.copy()
    second = top2_features.copy()
    first.columns = [
        f"top1_{name}" for name in FORECAST_FEATURE_NAMES
    ]
    second.columns = [
        f"top2_{name}" for name in FORECAST_FEATURE_NAMES
    ]
    signed = (
        top1_features.to_numpy() - top2_features.to_numpy()
    )
    differences = pd.DataFrame(
        np.concatenate([signed, np.abs(signed)], axis=1),
        columns=[
            *[
                f"top1_minus_top2_{name}"
                for name in FORECAST_FEATURE_NAMES
            ],
            *[
                f"top1_top2_absdiff_{name}"
                for name in FORECAST_FEATURE_NAMES
            ],
        ],
        dtype=np.float32,
    )
    pair = pd.DataFrame(
        compute_pair_features(
            top1_predictions,
            top2_predictions,
            contexts[:, 0],
            contexts[:, 1],
        ),
        columns=[
            f"pair_{name}" for name in PAIR_FEATURE_NAMES
        ],
    )
    identity = pd.DataFrame(
        {
            "top1_model": pd.Categorical.from_codes(
                top1, categories=models
            ),
            "top2_model": pd.Categorical.from_codes(
                top2, categories=models
            ),
        }
    )
    rank = pd.DataFrame(
        rank_features,
        columns=[
            f"pre_{name}" for name in RANK_FEATURE_NAMES
        ],
        dtype=np.float32,
    )
    frame = pd.concat(
        [
            lookback.reset_index(drop=True),
            first,
            second,
            differences,
            pair,
            identity,
            rank,
        ],
        axis=1,
    )
    missing = sorted(set(expected_columns).difference(frame))
    if missing:
        raise ValueError(
            f"Post feature mismatch: missing={missing}"
        )
    return frame.reindex(columns=expected_columns)


def evaluate_forecasts(
    spec: ConfigSpec,
    forecasts: np.ndarray,
    gift_eval_data: Path,
) -> dict[str, float]:
    from gift_eval.data import Dataset
    from gluonts.ev.metrics import (
        MAE,
        MAPE,
        MASE,
        MSE,
        MSIS,
        ND,
        NRMSE,
        RMSE,
        SMAPE,
        MeanWeightedSumQuantileLoss,
    )
    from gluonts.model import evaluate_model
    from gluonts.time_feature import get_seasonality

    os.environ["GIFT_EVAL"] = str(gift_eval_data)
    probe = Dataset(
        storage_env_var="GIFT_EVAL",
        name=spec.dataset_name,
        term=spec.term,
        to_univariate=False,
    )
    dataset = (
        probe
        if probe.target_dim == 1
        else Dataset(
            storage_env_var="GIFT_EVAL",
            name=spec.dataset_name,
            term=spec.term,
            to_univariate=True,
        )
    )
    metrics = [
        MSE(forecast_type="mean"),
        MSE(forecast_type="0.5"),
        MAE(forecast_type="0.5"),
        MASE(forecast_type="0.5"),
        MAPE(forecast_type="0.5"),
        SMAPE(forecast_type="0.5"),
        MSIS(),
        RMSE(forecast_type="mean"),
        NRMSE(forecast_type="mean"),
        ND(forecast_type="0.5"),
        MeanWeightedSumQuantileLoss(
            quantile_levels=QUANTILE_LEVELS
        ),
    ]
    result = evaluate_model(
        ArrayQuantilePredictor(
            spec.prediction_length, forecasts
        ),
        test_data=dataset.test_data,
        metrics=metrics,
        batch_size=512,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=int(get_seasonality(dataset.freq)),
    )
    return {
        column: float(result[column].iloc[0])
        for column in METRIC_COLUMNS
    }


def geometric_mean(values: np.ndarray) -> float:
    clean = values[(values > 0.0) & np.isfinite(values)]
    return float(np.exp(np.mean(np.log(clean))))


def main() -> None:
    args = parse_args()
    sys.path.insert(0, str(args.gift_eval_repo.resolve() / "src"))
    test_feature_root = (
        args.test_feature_root
        if args.test_feature_root is not None
        else args.artifact_root / "test_features"
    )
    test_prediction_root = (
        args.test_prediction_root
        if args.test_prediction_root is not None
        else args.artifact_root / "test_predictions"
    )

    manifest = json.loads(args.manifest.read_text())
    models = [str(value) for value in manifest["models"]]
    feature_columns = [
        str(value) for value in manifest["feature_columns"]
    ]
    categories = {
        key: [str(value) for value in values]
        for key, values in manifest["categories"].items()
    }
    specs = [
        spec_from_row(row) for row in manifest["configs"]
    ]
    if args.limit_configs is not None:
        specs = specs[: args.limit_configs]
    biases = load_biases(args.class_bias, models)

    model_models = json.loads(
        (args.pre_model_root / "models.json").read_text()
    )
    if model_models != models:
        raise ValueError("Pre model/manifest vocabulary mismatch")

    post_booster = xgb.Booster()
    post_booster.load_model(args.post_model)
    post_booster.set_param({"device": args.device})
    expected_post_columns = list(post_booster.feature_names or [])
    if not expected_post_columns:
        raise ValueError("Post model does not contain feature names")

    reference = None
    if not args.skip_reference_check:
        reference = pd.read_csv(args.reference_metrics)
        reference = reference[
            (reference["split"] == "test")
            & (reference["method"] == args.reference_method)
        ].set_index("dirname")
        if len(reference) != 97:
            raise ValueError(
                f"Expected 97 reference rows for {args.reference_method}, "
                f"found {len(reference)}"
            )

    output_rows = []
    normalized_mase = []
    normalized_crps = []
    for position, spec in enumerate(specs, start=1):
        print(
            f"[{position}/{len(specs)}] {spec.dataset_config}",
            flush=True,
        )
        lookback = load_lookback_frame(
            test_feature_root,
            spec,
            feature_columns,
            categories,
        )
        pre_booster = xgb.Booster()
        pre_booster.load_model(
            args.pre_model_root
            / f"{spec.bucket.replace('|', '_')}.ubj"
        )
        pre_booster.set_param({"device": args.device})
        top1, top2, rank_features = predict_pre_ranking(
            pre_booster,
            lookback,
            len(models),
            biases[spec.bucket],
        )
        top1_predictions = load_selected_predictions(
            test_prediction_root, models, spec, top1
        )
        top2_predictions = load_selected_predictions(
            test_prediction_root, models, spec, top2
        )
        top1_features = load_selected_window_features(
            args.all_model_feature_root, models, spec, top1
        )
        top2_features = load_selected_window_features(
            args.all_model_feature_root, models, spec, top2
        )
        contexts = load_context_end_and_scale(
            args.gift_eval_data, spec
        )
        post_frame = build_post_frame(
            lookback,
            top1_features,
            top2_features,
            top1_predictions,
            top2_predictions,
            top1,
            top2,
            rank_features,
            contexts,
            models,
            expected_post_columns,
        )
        best_iteration = getattr(
            post_booster, "best_iteration", None
        )
        iteration_range = (
            (0, int(best_iteration) + 1)
            if best_iteration is not None
            else (0, 0)
        )
        post_matrix = xgb.DMatrix(
            post_frame, enable_categorical=True
        )
        if args.post_output_mode == "binary_softmax":
            post_logits = np.asarray(
                post_booster.predict(
                    post_matrix,
                    output_margin=True,
                    iteration_range=iteration_range,
                ),
                dtype=np.float32,
            ).reshape(spec.rows, 2)
            top1_weights = softmax(post_logits)[:, 1]
        else:
            top1_weights = np.clip(
                np.asarray(
                    post_booster.predict(
                        post_matrix,
                        iteration_range=iteration_range,
                    ),
                    dtype=np.float32,
                ).reshape(spec.rows),
                0.0,
                1.0,
            )
        forecasts = (
            top1_weights[:, None, None] * top1_predictions
            + (1.0 - top1_weights[:, None, None]) * top2_predictions
        ).astype(np.float32)
        metrics = evaluate_forecasts(
            spec, forecasts, args.gift_eval_data
        )

        if reference is not None:
            expected = reference.loc[spec.dirname]
            for metric_name, reference_column in {
                "MASE[0.5]": "mase",
                "mean_weighted_sum_quantile_loss": "crps",
            }.items():
                if not math.isclose(
                    metrics[metric_name],
                    float(expected[reference_column]),
                    rel_tol=0.0,
                    abs_tol=REFERENCE_METRIC_ATOL,
                ):
                    raise AssertionError(
                        f"{spec.dirname}/{metric_name}: "
                        f"{metrics[metric_name]} != "
                        f"{expected[reference_column]}"
                    )
        normalized_mase.append(
            metrics["MASE[0.5]"] / spec.normalizer_mase
        )
        normalized_crps.append(
            metrics["mean_weighted_sum_quantile_loss"]
            / spec.normalizer_crps
        )
        output_rows.append(
            {
                "dataset": official_dataset_name(spec.dataset_config),
                "model": args.model_name,
                **{
                    f"eval_metrics/{name}": metrics[name]
                    for name in METRIC_COLUMNS
                },
                "domain": spec.domain,
                "num_variates": spec.num_variates,
            }
        )

    output = pd.DataFrame(output_rows).reindex(
        columns=OUTPUT_COLUMNS
    )
    args.output.parent.mkdir(parents=True, exist_ok=True)
    output.to_csv(args.output, index=False)

    if args.limit_configs is None:
        if output.shape != (97, 15):
            raise AssertionError(
                f"Expected shape (97, 15), got {output.shape}"
            )
        composite = 0.5 * (
            geometric_mean(np.asarray(normalized_mase))
            + geometric_mean(np.asarray(normalized_crps))
        )
        if not math.isclose(
            composite,
            args.expected_composite,
            rel_tol=0.0,
            abs_tol=2e-7,
        ):
            raise AssertionError(
                f"Composite {composite} != "
                f"{args.expected_composite}"
            )
        print(f"Composite={composite:.12f}")
    print(f"Wrote {args.output} with shape {output.shape}")


## Run the full reproduction

Set `limit_configs` in `NOTEBOOK_ARGS` to a positive integer for a shorter local run.


In [ ]:
main()
